# اليوم الثالث: اختر عتبة قرار تستطيع تبريرها
**SDA-DSC-211 · Tamweel Lite · Day 3 · 60 minutes**  
إعداد وتقديم **ميعاد المري | Meaad Al-Marri**

ستقارن وزن الفئة وإعادة أخذ العينات بخط أساس غير موزون، ثم تحوّل درجات النموذج إلى **إشارات مراجعة تعليمية** وفق خسارة الأخطاء وسعة الفريق. لا يوجد نموذج مطلوب أن يفوز، ولا انخفاض مضمون للخسارة بعد تقييد السعة.

**تسليمك:** `cost_curve.png`، و`threshold_metrics.json`، وبطاقة `DECISION_CARD.md` بأرقام تشغيلك وتفسيرك. افتح جلسة **CPU مجانية جديدة** وشغّل **Runtime → Run all**. لا يلزم دفتر سابق أو ربطDrive أو اشتراك. احتفظ بالخانات فارغة أولًا، ثم املأها وأعد خليتي التفسير والحفظ.

| الوقت | مهمتك |
|---|---|
| 0–10 | اقرأ الخسارة والسعة وجهّز التدريب |
| 10–20 | أنشئ تنبؤاتOOF للاستراتيجيات الثلاث |
| 20–35 | قارن المقاييس وارسم منحنى الخسارة |
| 35–45 | طبّق السعة على كل فترة |
| 45–50 | قارن أثر إعادة العينات |
| 50–55 | افحص الإنذار الخاطئ حسب المنطقة |
| 55–60 | اكتب بطاقة قرارك واحفظها |

This lab uses fictional applications. A flag means a simulated review, not automatic approval or refusal. Costs are educational loss units, not fees or grade deductions.

## 1. جهّز بيئتك | Setup
يفحص الإعداد الإصدارات وبصمات البيانات والأدوات. إذا طُلبت إعادة الجلسة، اختر **Restart session** ثمRun all. الوضعFAST الافتراضي يثبت80 شجرة لكل نموذج وخيطَيCPU؛ FULL اختياري يثبت160. لا نعيد البحث عن معاملات بعد رؤية نتائجOOF اليوم.

تعاد الاستفادة من تقسيم اليوم الثاني. يبقى عدد الأشجار ثابتًا قبل المقارنة حتى تدرس أثر معالجة عدم التوازن وحدها، دون اعتماد على ملفات محفوظة في جلسة سابقة.

In [ ]:
from pathlib import Path
import hashlib, importlib.util, os, runpy, sys, time, urllib.request, urllib.error

RUN_STARTED = time.perf_counter()
COURSE_REVISION = "fe0c0204e6076a7ac2139b7336485a097343fb8a"
MANIFEST_SHA256 = "86d15e813caf833b7d86512808176388d4123dc3c01c6525ee849a76e8322c59"
SETUP_SHA256 = "79608f8f9559d9e6eb09d51ee541ee12d7e9262ac57d1ca3a948c8ecb14f0f35"
IN_COLAB = importlib.util.find_spec("google") is not None and importlib.util.find_spec("google.colab") is not None
ROOT = Path("/content/tamweel") if IN_COLAB else Path(os.environ.get("TAMWEEL_PROJECT_DIR", ".")).resolve()
setup_path = ROOT / "scripts/setup_colab.py"
setup_path.parent.mkdir(parents=True, exist_ok=True)
if not setup_path.exists() or hashlib.sha256(setup_path.read_bytes()).hexdigest() != SETUP_SHA256:
    setup_url = f"https://raw.githubusercontent.com/almiyead-rgb/sda-dsc-211-student-template/{COURSE_REVISION}/scripts/setup_colab.py"
    for attempt in range(3):
        try:
            with urllib.request.urlopen(setup_url, timeout=45) as response:
                setup_bytes = response.read(200000)
            if hashlib.sha256(setup_bytes).hexdigest() != SETUP_SHA256:
                raise ValueError("Setup checksum mismatch. Reopen the released notebook.")
            setup_path.write_bytes(setup_bytes)
            break
        except (urllib.error.URLError, TimeoutError):
            if attempt == 2:
                raise RuntimeError("Download unavailable. Retry setup or upload the unchanged project files; see READINESS_GUIDE.")
            time.sleep(attempt + 1)
environment = runpy.run_path(str(setup_path))["prepare"](ROOT, COURSE_REVISION, MANIFEST_SHA256)
FAST_MODE, FULL_MODE, SEED, N_JOBS = True, False, 211, 2

from setup_colab import fetch_verified
DAY2_REVISION = "1e1da4acbee8941bef07245b259fb6c27ced4ad7"
DAY2_SHA256 = "d2fd0353330c3629bdaed6c570528792955fce232540a85d33532a9432522f43"
fetch_verified(f"https://raw.githubusercontent.com/almiyead-rgb/sda-dsc-211-student-template/{DAY2_REVISION}/scripts/day2_validation.py",
               ROOT / "scripts/day2_validation.py", DAY2_SHA256)
LAB_REVISION = "c2dc52e129878d0d487db1874891c658de7c4d0d"
SUPPORT_HASHES = {'scripts/day3_decision.py': '40f176633c91fc530db222978bc44495a401b888b1ce448728948f7ae0ca9e12', 'data/day3_oof_example.csv': '659952bef85ac3b7ceb7ab84eb2102de6c9b433a1ba122cd57df96581baf0eb3', 'data/day3_oof_example.json': '920cf9830043afcc83f8c276c8945677296eb3c49a6e69326d47f6080a9e7fad'}
for relative, digest in SUPPORT_HASHES.items():
    fetch_verified(f"https://raw.githubusercontent.com/almiyead-rgb/sda-dsc-211-student-template/{LAB_REVISION}/{relative}",
                   ROOT / relative, digest)
print("Day 3 setup verified | إعداد اليوم الثالث جاهز")

## 2. ثبّت سياسة التمرين | Loss and capacity
الهدف وقوع حدث تعثر **خلال90 يومًا بعد الطلب**، وليس بلوغ90 يوم تأخر. البيانات كلها اصطناعية. نقرأ التدريب فقط؛ لا يستخدم التحدي للاختيار أو التقييم اليوم، وإن نزّله الإعداد للتحقق من حزمة البيانات.

| النتيجة | معناها في التمرين | خسارتها |
|---|---|---:|
| TP | إشارة لحدث تعثر لاحق | 0 |
| FN | تعثر لم تُرفع له إشارة | 10 |
| FP | إشارة لحالة سليمة | 1 |
| TN | حالة سليمة بلا إشارة | 0 |

**الخسارة المرصودة =10×FN +1×FP**. هذه وحدات تعليمية تعكس تفاوت الأخطاء؛ ليست ريالات فعلية، ولا رسوم أدوات، ولا خصمًا من الدرجة. تكلفة المراجعة وفعاليتها غير ممثلتين في هذا النموذج المبسّط.

**السعة:** لا تتجاوز الإشارات12% من طلبات **كل فترة تحقق**، بعد التقريب إلى العدد الصحيح الأدنى. تطبيق السعة على المجموع فقط قد يخفي ازدحام فترة بعينها. اترك السياسة المعتمدة في التجربة الأساسية؛ ستفحص حساسيةFN لاحقًا دون تغييرها.

In [ ]:
import json, platform, zipfile
from dataclasses import asdict, replace
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve
from IPython.display import display
from data_checks import validate_frame
from day2_validation import deduplicate_applications, make_time_group_split
from day3_decision import (STRATEGIES, TARGET, DecisionConfig, CostPolicy, generate_oof,
    validate_oof, threshold_sweep, select_threshold, flag_at, period_audit, region_audit,
    reflection_check, load_example, TrainingBudgetExpired)

contract = json.loads((ROOT / "data/data_contract.json").read_text(encoding="utf-8"))
train, duplicates = deduplicate_applications(pd.read_csv(ROOT / "data/tamweel_train.csv"))
validate_frame(train, contract, "train")
FAST_MODE, FULL_MODE = True, False
assert FAST_MODE != FULL_MODE
config = DecisionConfig(trees=80 if FAST_MODE else 160)
policy = CostPolicy(**contract["decision_policy"])
MODEL_FOR_DECISION = "weighted"  # Declared before comparison; not a claim of superiority.
USE_EXAMPLE = False  # True is discussion only, never your submission.
ASSESSMENT_MODE = os.environ.get("ASSESSMENT_MODE", "0") == "1"
assert MODEL_FOR_DECISION in STRATEGIES
ARTIFACTS, REPORTS = ROOT / "artifacts", ROOT / "reports"
ARTIFACTS.mkdir(exist_ok=True); REPORTS.mkdir(exist_ok=True)
folds = make_time_group_split(train, horizon_days=contract["target"]["horizon_days"])
display(pd.DataFrame([{"training_rows": len(train), "positive_rate": train[TARGET].mean(),
    "FN_loss_units": policy.false_negative_cost, "FP_loss_units": policy.false_positive_cost,
    "period_capacity_fraction": policy.max_flag_fraction, "trees_per_model": config.trees}]))
display(pd.DataFrame([fold["audit"] for fold in folds]))

## 3. عالج التدريب فقط | Weighting and resampling
نستخدمLightGBM بالإعدادات الأساسية نفسها والتقسيم نفسه. لكل طية يحترم التدريب نضج90 يومًا ويفصل عملاء التحقق. تبقى المعرفات والتاريخ والهدف خارج المدخلات.

| الاستراتيجية | ماذا تغيّر؟ |
|---|---|
| `unweighted` | مرجع بلا تغيير أوزان أو نسخ صفوف |
| `weighted` | `scale_pos_weight = negatives / positives` من تدريب الطية فقط |
| `oversampled` | نسخ عشوائي للفئة الأقل داخل التدريب حتى توازن الفئتين |

يتعلم الوسيط من **صفوف التدريب الأصلية** قبل نسخها. لا يُعاد أخذ عينات التحقق؛ تحتفظ بنسبة الموجب الفعلية. الأوزان ليست مساوية تلقائيًا لنسبة خسارةFN/FP؛ الأولى تعدّل التدريب، والثانية تختار القرار بعده.

قد تتغير معايرة الدرجات بعد الترجيح أو إعادة العينات؛ لا تفسّر الدرجة0.7 كاحتمال موثوق70% دون فحص. ستدرس المعايرة في اليوم الرابع. راجع [أوزانLightGBM](https://lightgbm.readthedocs.io/en/v4.6.0/Parameters.html#scale_pos_weight).

الحد120 ثانية للتدريب تعاوني بين الأشجار والنماذج، وقد يتجاوز قليلًا بكلفة الإنهاء. عند نفاده لا نقبل نتيجةOOF ناقصة؛ نعرض مثالًا ثابتًا موسومًا للنقاش، وتصبح البطاقة **غير صالحة للتسليم كتجربة شخصية**. في وضع التقييم يُمنع المثال ويُطلب تشغيل فعلي.

In [ ]:
fallback_reason = None
if not USE_EXAMPLE:
    try:
        oof, model_report, provenance = generate_oof(train, contract, config)
    except TrainingBudgetExpired as exc:
        if ASSESSMENT_MODE:
            raise
        USE_EXAMPLE, fallback_reason = True, str(exc)
if USE_EXAMPLE:
    oof, example_info = load_example(ROOT / "data/day3_oof_example.csv",
        ROOT / "data/day3_oof_example.json", train, folds, assessment_mode=ASSESSMENT_MODE)
    model_report = pd.DataFrame(example_info["model_report"])
    provenance = {"source": "EDUCATIONAL_EXAMPLE", "example_metadata": example_info,
                  "fallback_reason": fallback_reason or "Learner selected discussion example"}
SOURCE = oof.source.iloc[0]
effective_config = example_info["config"] if USE_EXAMPLE else asdict(config)
coverage = validate_oof(oof, train, folds)
print("LIVE — your CPU run" if SOURCE == "LIVE" else
      "EDUCATIONAL_EXAMPLE — discussion only; EXAMPLE_ONLY_NOT_SUBMITTABLE")
display(pd.DataFrame([coverage]))
display(model_report[["strategy", "fold", "original_training_rows", "fitted_rows",
    "scale_pos_weight", "training_positive_rate", "fitted_positive_rate", "average_precision",
    "train_seconds", "source"]].round(4))
selected = oof[oof.strategy.eq(MODEL_FOR_DECISION)].copy()
assert selected.application_id.is_unique
print(f"Decision strategy declared before comparison: {MODEL_FOR_DECISION}")

## 4. لا تدعAccuracy تخدعك | Ranking and operating point
كل استراتيجية تغطي5,039 طلبًا لاحقًا، وتبقى4,961 عينة تمهيدية بلاOOF. التغطية50.39% من كل التدريب و100% من الصفوف المؤهلة. لا تستبدل المفقود بتنبؤات تدريب.

نبلغ عنROC-AUC و**Average Precision (AP)** على احتمالاتOOF المجمعة للصفوف نفسها. AP هو التعريف المعتمد لـPR-AUC هنا، وليس مساحة شبه المنحرف. هذا المقياس المجمّع ليس متوسطAP للطيات الذي قرأته في اليوم الثاني؛ المقام وطريقة التجميع معلنان.

راقب أيضًا نسبة الموجب. مرجع «لا إشارات للجميع» قد يحققAccuracy مرتفعة معRecall يساوي صفرًا. [تعريفAverage Precision](https://scikit-learn.org/1.6/modules/generated/sklearn.metrics.average_precision_score.html).

In [ ]:
comparison_rows = []
for strategy, group in oof.groupby("strategy", sort=False):
    sweep = threshold_sweep(group, policy)
    default = sweep.loc[sweep.threshold.eq(.5)].iloc[0]
    comparison_rows.append({"strategy": strategy, "rows": len(group), "positive_rate": group[TARGET].mean(),
        "pooled_roc_auc": roc_auc_score(group[TARGET], group.probability),
        "pooled_ap": average_precision_score(group[TARGET], group.probability),
        "accuracy_at_0_5": default.accuracy, "recall_at_0_5": default.recall,
        "precision_at_0_5": default.precision, "flags_at_0_5": default.flagged,
        "loss_at_0_5": default.loss_units, "capacity_ok_at_0_5": default.capacity_feasible, "source": SOURCE})
comparison = pd.DataFrame(comparison_rows)
display(comparison.round(4))
no_flag_reference = {"rule": "No flags for anyone", "rows": len(selected),
    "accuracy": 1-selected[TARGET].mean(), "recall": 0., "precision": None,
    "loss_units": int(selected[TARGET].sum()) * policy.false_negative_cost}
display(pd.DataFrame([no_flag_reference]))

plt.rcParams.update({"font.size": 11, "axes.spines.top": False, "axes.spines.right": False,
                     "figure.dpi": 120, "axes.grid": True, "grid.alpha": .2})
colors = {"unweighted": "#2667A0", "weighted": "#BA5A17", "oversampled": "#18816A"}
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.3), constrained_layout=True)
for strategy, style in zip(STRATEGIES, ["-", "--", "-."]):
    group = oof[oof.strategy.eq(strategy)]
    fpr, tpr, _ = roc_curve(group[TARGET], group.probability)
    precision, recall, _ = precision_recall_curve(group[TARGET], group.probability)
    row = comparison.set_index("strategy").loc[strategy]
    axes[0].plot(fpr, tpr, color=colors[strategy], linestyle=style, label=f"{strategy}: {row.pooled_roc_auc:.3f}")
    axes[1].plot(recall, precision, color=colors[strategy], linestyle=style, label=f"{strategy}: AP {row.pooled_ap:.3f}")
axes[0].plot([0,1], [0,1], color="#777777", linestyle=":", label="Random ranking")
axes[1].axhline(selected[TARGET].mean(), color="#777777", linestyle=":", label=f"Prevalence: {selected[TARGET].mean():.1%}")
axes[0].set(title="ROC", xlabel="False positive rate", ylabel="Recall")
axes[1].set(title="Precision–recall", xlabel="Recall", ylabel="Precision")
for ax in axes:
    ax.set_xlim(0,1); ax.set_ylim(0,1.02); ax.legend(fontsize=8)
fig.suptitle(f"{SOURCE} · pooled OOF · {len(selected):,} synthetic requests")
fig.savefig(ARTIFACTS / "day3_roc_pr.png", dpi=160)
plt.show()

## 5. جرّب كل قواعد العتبة الممكنة | Threshold sweep
نرفع إشارة عندما **الدرجة ≥ العتبة**. نفحص جميع الدرجات المميزة، ونضيف0.5 صراحة للمقارنة، وقاعدة لا ترفع أي إشارة. لا نحصر البحث في شبكة خشنة قد تفوّت أفضل قاعدة.

عند تعادل الدرجات تبقى الطلبات المتعادلة معًا؛ لا نكسر التعادل بالمعرّف لملء السعة. إذا تساوت الخسارة نفضّل إشارات أقل، ثم عتبة أعلى. قد تبقى سعة غير مستخدمة.

نختار عتبة بأقل خسارة دون قيد، وأخرى بأقل خسارة ضمن12% **في كل فترة**. اختيار العتبة علىOOF أفضل من تنبؤات التدريب، لكننا ما زلنا نستخدم أهدافOOF للاختيار؛ لذلك خسارتها المختارة **تقدير تطوير متفائل، لا اختبارًا نهائيًا مستقلًا**. التحدي يبقى مغلقًا. [فصل تدريب النموذج عن ضبط القرار](https://scikit-learn.org/stable/modules/classification_threshold.html).

In [ ]:
cost_curve = threshold_sweep(selected, policy)
unconstrained = select_threshold(cost_curve, constrained=False)
chosen = select_threshold(cost_curve, constrained=True)
default = cost_curve.loc[cost_curve.threshold.eq(.5)].iloc[0].to_dict()
operating_points = pd.DataFrame([{**point, "rule": name} for name, point in
    [("Default 0.5", default), ("Minimum loss, no capacity limit", unconstrained),
     ("Minimum loss, every-period capacity", chosen)]])
display(operating_points[["rule", "threshold", "tp", "fp", "fn", "tn", "recall", "precision",
    "flagged", "flag_fraction", "loss_units", "loss_units_per_10000", "capacity_feasible"]].round(4))
assert unconstrained["loss_units"] <= default["loss_units"]
if default["capacity_feasible"]:
    assert chosen["loss_units"] <= default["loss_units"]
periods = period_audit(selected, chosen["threshold"], policy)
assert periods.within_capacity.all()
delta = chosen["loss_units"] - default["loss_units"]
print(f"Constrained minus default loss: {delta:+.0f} educational units. Default feasible: {default['capacity_feasible']}")
print("Exact rule to preserve in JSON:", repr(chosen["threshold"]), "with >=")

fig, axes = plt.subplots(2, 1, figsize=(10, 6.5), sharex=True, constrained_layout=True)
axes[0].plot(cost_curve.threshold, cost_curve.loss_units_per_10000, color="#2667A0")
axes[0].fill_between(cost_curve.threshold, 0, cost_curve.loss_units_per_10000,
    where=cost_curve.capacity_feasible, color="#18816A", alpha=.12, label="Feasible in every period")
for label, point, marker, color in [("0.5", default, "s", "#555555"),
    ("Unconstrained", unconstrained, "o", "#BA5A17"), ("Capacity-constrained", chosen, "D", "#18816A")]:
    axes[0].scatter([point["threshold"]], [point["loss_units_per_10000"]], marker=marker, color=color, s=50,
        label=f"{label}: {point['loss_units_per_10000']:.0f} units / 10,000")
axes[0].set(ylabel="Observed loss units / 10,000", ylim=(0, cost_curve.loss_units_per_10000.max()*1.08))
axes[0].legend(fontsize=8)
axes[1].plot(cost_curve.threshold, cost_curve.flag_fraction*100, label="All periods combined", color="#2667A0")
axes[1].plot(cost_curve.threshold, cost_curve.max_period_flag_fraction*100, linestyle="--",
    label="Most loaded period", color="#BA5A17")
axes[1].axhline(policy.max_flag_fraction*100, color="#444444", linestyle=":", label="12% period ceiling")
axes[1].axvline(chosen["threshold"], color="#18816A", linestyle=":", label="Chosen threshold")
axes[1].set(xlabel="Score threshold (flag when score ≥ threshold)", ylabel="Requests flagged (%)", ylim=(0,102), xlim=(0,1.02))
axes[1].legend(fontsize=8)
fig.suptitle(f"{SOURCE} · {MODEL_FOR_DECISION} · OOF development selection, not final test")
fig.savefig(ARTIFACTS / "cost_curve.png", dpi=160)
plt.show()

## 6. افحص السعة وحساسية الافتراض | Capacity and sensitivity
جدول كل فترة هو دليل السعة الفعلي. النسبة المجمعة وحدها لا تكفي. العتبة المحفوظة دقيقة؛ تقريبها عند التطبيق قد يغيّر عدد الإشارات، لذا استخدم القيمة الكاملة منJSON.

أقل خسارة دون قيد لا تتجاوز خسارة0.5 علىOOF لأن0.5 ضمن المرشحين. أما العتبة المقيدة فقد ترفع الخسارة إذا كانت0.5 غير قابلة للتنفيذ. لا تكتب «توفيرًا» عندما يظهر فرق موجب.

غيّر افتراض خسارةFN إلى8 و12 مع إبقاءFP=1 والسعة12%. هذه **سيناريوهات ±20% وليست فترات ثقة**. المقارنة تبيّن حساسية القرار؛ لا تغيّر إعدادات التجربة الأساسية تلقائيًا. السعة التي تحققت تاريخيًا لا تضمن السعة في دفعة مستقبلية؛ راقب حجم الطابور وتغير توزيع الدرجات.

In [ ]:
display(periods)
sensitivity_rows = []
for fn_cost in [8., 10., 12.]:
    scenario_policy = replace(policy, false_negative_cost=fn_cost)
    scenario = select_threshold(threshold_sweep(selected, scenario_policy))
    sensitivity_rows.append({"FN_loss_units": fn_cost, "FP_loss_units": policy.false_positive_cost,
        "capacity_fraction": policy.max_flag_fraction, "threshold": scenario["threshold"],
        "flagged": scenario["flagged"], "recall": scenario["recall"],
        "loss_units": scenario["loss_units"], "capacity_feasible": scenario["capacity_feasible"], "source": SOURCE})
sensitivity = pd.DataFrame(sensitivity_rows)
display(sensitivity.round(4))
print("Theoretical threshold 1/11 ≈ 0.0909 requires calibrated probabilities, this loss matrix and no capacity limit.")
print("Do not impose that formula on today's weighted raw scores.")

## 7. بلّغ عن الفروق بين المناطق | Descriptive group audit
بعد تثبيت العتبة نحسب **معدل الإنذار الخاطئ =FP ÷ عدد الحالات السليمة** في كل منطقة اصطناعية. اعرض المقام، وعدد الإشارات، وRecall أيضًا. `other` تعني عدم تفعيل المؤشرات الجغرافية الثلاثة؛ لا تنسبها إلى منطقة حقيقية محددة.

الفجوة هي أكبر معدل ناقص أصغره، بوحدة **نقطة مئوية**. نقص المقام ينتج قيمة غير متاحة، لا صفرًا. تُوسم المجموعة إذا كان عدد الحالات السليمة أقل من30. هذا حد تنبيه تعليمي لا اختبار دلالة.

الاختلاف لا يثبت سببية المنطقة أو انحيازًا قانونيًا، والفجوة الصغيرة لا تثبت العدالة. العملاء المتكررون والطيات مترابطة، ولم نحسب فترات ثقة. لا تخفِ الفارق أو تغيّر العتبة لكل منطقة لإخفائه؛ اذكر ما يحتاج إلى مراجعة.

In [ ]:
regions, gap_info = region_audit(selected, chosen["threshold"])
display(regions.round(4))
print("FPR gap (percentage points):", gap_info["fpr_gap_percentage_points"])
fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
x = np.arange(len(periods))
axes[0].bar(x-.18, periods.capacity, .36, color="#D3D9DE", edgecolor="#555555", label="Capacity, rounded down")
axes[0].bar(x+.18, periods.flagged, .36, color="#2667A0", label="Actual flags")
axes[0].set(xticks=x, xticklabels=[f"Period {n}" for n in periods.fold], ylabel="Requests", title="Capacity by period")
axes[0].set_ylim(0, periods.capacity.max()*1.4)
axes[0].legend(fontsize=8)
axes[1].bar(regions.region, regions.false_positive_rate*100, color="#BA5A17")
axes[1].set(ylabel="Good requests flagged (%)", title="Regional FPR · descriptive, no intervals")
for i, row in enumerate(regions.itertuples()):
    if pd.notna(row.false_positive_rate):
        axes[1].text(i, row.false_positive_rate*100+.15, f"{row.false_positive_rate:.1%}\nn={row.negatives}", ha="center", fontsize=9)
axes[1].set_ylim(0, max(10, regions.false_positive_rate.max()*140))
fig.suptitle(f"{SOURCE} · one shared threshold · synthetic requests only")
fig.savefig(ARTIFACTS / "day3_capacity_regions.png", dpi=160)
plt.show()

## 8. اكتب حجتك | Your decision
الخلية التالية تبقى عملك أنت. استخدم أرقامك في سبب اختيار العتبة، ثم اشرح المقايضة بين الخسارة والسعة والفجوة الجغرافية وحدود النتيجة. **لا يكفي نسخ رقم العتبة.**

أجب أيضًا: لماذا تخدعكAccuracy عند ندرة التعثر؟ ولماذا تضبط العتبة علىOOF بدل تنبؤات التدريب أو التحدي؟ اكتمال الحقول لا يمنح درجة ولا يضمن صحة التفسير.

In [ ]:
responses = {"threshold_reason": "", "cost_capacity_tradeoff": "", "regional_gap": "",
    "limitations": "", "why_accuracy_misleads": "", "why_oof": ""}
checkpoint = reflection_check(responses, SOURCE)
print(checkpoint["status"], "| missing:", checkpoint["missing"])

## 9. صدّر البطاقة والأدلة | Export
الخلية تنشئ `reports/DECISION_CARD.md` بالأرقام الفعلية وخانات تفسيرك، وتحفظCSV وJSON والرسوم. إذا كانت إجاباتك فارغة تبقى البطاقة مسودة. وإذا استخدمت المثال تظهر عبارة **مثال تعليمي غير صالح للتسليم كتجربة شخصية** في البطاقة وكل الجداول، ويُمنع اعتباره تشغيلًا ناجحًا لك.

نزّل `day3_artifacts.zip` من **Files → tamweel → artifacts** وافكّه في جذر مستودعك؛ ستجد المجلدين`artifacts/` و`reports/` جاهزين. احفظ الدفتر من **File → Download → Download .ipynb** باسم `notebooks/03_cost_sensitive_decision.ipynb`. لا ترفعZIP وحده وتفترض أن محتوياته ظهرت في المجلدات.

`TECHNICAL_READY` يعني اكتمال التشغيل الفعلي والفحوص؛ `LEARNER_WORK_REQUIRED` يعني بقاء تفسيرك. `EXAMPLE_ONLY_NOT_SUBMITTABLE` للنقاش فقط. لا توجد درجة آلية.

In [ ]:
def json_safe(value):
    if isinstance(value, dict): return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)): return [json_safe(v) for v in value]
    if isinstance(value, np.generic): value = value.item()
    if isinstance(value, float) and not np.isfinite(value): return None
    return value

checkpoint = reflection_check(responses, SOURCE)
status = "TECHNICAL_READY" if SOURCE == "LIVE" else "EXAMPLE_ONLY_NOT_SUBMITTABLE"
pooled_ap = float(average_precision_score(selected[TARGET], selected.probability))
threshold_metrics = {"source": SOURCE, "technical_status": status, "strategy": MODEL_FOR_DECISION,
    "policy": asdict(policy), "unit": "educational_loss_unit", "comparison_operator": ">=",
    "capacity_scope": "every validation period; floor(fraction * period_rows)",
    "chosen": chosen, "unconstrained": unconstrained, "default_0_5": default,
    "loss_change_vs_default": delta, "pooled_oof_ap": pooled_ap, "regional_audit": gap_info,
    "coverage": coverage, "selection_status": "OOF development selection; not an independent final test"}
membership = train[["application_id", "application_date"]].copy()
membership["role"], membership["fold"] = "warmup_no_oof", 0
for fold in folds:
    membership.loc[fold["valid"], "role"] = "eligible_oof"
    membership.loc[fold["valid"], "fold"] = fold["audit"]["fold"]
membership["source"] = SOURCE
decisions = selected.copy()
decisions["review_flag"] = flag_at(selected.probability, chosen["threshold"]).astype(int)
csv_outputs = {"day3_model_report.csv": model_report, "day3_model_comparison.csv": comparison,
    "day3_oof_predictions.csv": oof, "day3_oof_coverage.csv": membership, "threshold_sweep.csv": cost_curve,
    "day3_period_capacity.csv": periods, "day3_region_audit.csv": regions,
    "day3_cost_sensitivity.csv": sensitivity, "day3_review_flags.csv": decisions}
for filename, table in csv_outputs.items(): table.to_csv(ARTIFACTS / filename, index=False)
run = {"technical_status": status, "learner_status": checkpoint["status"], "source": SOURCE,
    "requested_config": asdict(config), "effective_config": effective_config, "assessment_mode": ASSESSMENT_MODE,
    "data_revision": COURSE_REVISION, "lab_revision": LAB_REVISION, "support_sha256": SUPPORT_HASHES,
    "day2_revision": DAY2_REVISION, "day2_sha256": DAY2_SHA256, "manifest_sha256": MANIFEST_SHA256,
    "training_data_sha256": hashlib.sha256((ROOT / "data/tamweel_train.csv").read_bytes()).hexdigest(),
    "python": platform.python_version(), "elapsed_seconds": round(time.perf_counter()-RUN_STARTED,2),
    "coverage": coverage, "challenge_used_for_modeling": False, "no_automatic_grade": True}
if IN_COLAB:
    import resource
    run["peak_process_memory_mib"] = round(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss/1024,1)
json_outputs = {"threshold_metrics.json": threshold_metrics, "day3_provenance.json": provenance,
    "day3_reflection.json": {"source": SOURCE, "responses": responses, "checkpoint": checkpoint}, "day3_run.json": run}
for filename, payload in json_outputs.items():
    (ARTIFACTS / filename).write_text(json.dumps(json_safe(payload), ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")

answer = lambda key: responses[key].strip() or "[اكتب تفسيرك من نتائجك هنا]"
rate_text = lambda value: f"{value:.2%}" if pd.notna(value) else "غير معرّفة لغياب المقام"
card_status = ("مثال تعليمي غير صالح للتسليم كتجربة شخصية" if SOURCE != "LIVE" else
               "مسودة — أكمل تفسيرك" if checkpoint["missing"] else "جاهزة للمراجعة؛ لا تعني اعتمادًا أو درجة")
card = f"""# بطاقة قرارك — Tamweel Lite

**الحالة:** {card_status}
**مصدر الأرقام:** {SOURCE} · **الاستراتيجية:** {MODEL_FOR_DECISION} · **الصفوف:** {len(selected):,} OOF

**المهمة:** الفئة الموجبة `default_within_90d=1` تعني حدث تعثر اصطناعي خلال90 يومًا بعد الطلب. كل طية تحقق طلبات لاحقة، وتستبعد عملاءها من التدريب وتشترط نضج نتيجة التدريب قبل بدايتها. المعرّفات والتاريخ خارج المدخلات.

| الدليل | القيمة |
|---|---:|
| العتبة المقيدة، بالقيمة الكاملة | {repr(chosen['threshold'])} |
| عتبة أقل خسارة دون قيد | {repr(unconstrained['threshold'])} |
| Recall | {rate_text(chosen['recall'])} |
| Precision | {rate_text(chosen['precision'])} |
| AP مجمع منOOF | {pooled_ap:.4f} |
| الإشارات | {chosen['flagged']:,} من {len(selected):,} |
| FN / FP | {chosen['fn']} / {chosen['fp']} |
| الخسارة التعليمية | {chosen['loss_units']:.0f} وحدة |
| خسارة0.5 | {default['loss_units']:.0f} وحدة؛ ضمن السعة: {default['capacity_feasible']} |
| التغير عن0.5 | {delta:+.0f} وحدة؛ الموجب زيادة |
| خسارة لكل10,000 طلب، تطبيع حسابي | {chosen['loss_units_per_10000']:.2f} وحدة |
| فجوة معدل الإنذار الخاطئ بين المناطق | {gap_info['fpr_gap_percentage_points']:.3f} نقطة مئوية |

**القاعدة:** درجة ≥ {repr(chosen['threshold'])} تعني إشارة مراجعة داخل التمرين؛ غير ذلك بلا إشارة. لا تتخذ موافقة أو رفض تمويل حقيقي. احفظ الدقة الكاملة؛ تقريب العتبة قد يغيّر حجم الطابور.

**السياسة:** FN={policy.false_negative_cost:g} وFP={policy.false_positive_cost:g} وحدات تعليمية، وسعة {policy.max_flag_fraction:.0%} لكل فترة بعد التقريب لأسفل. ليست ريالات فعلية أو رسوم أدوات أو خصمًا من الدرجة.

**دليل السعة:** {', '.join(f"الفترة {int(row.fold)}: {int(row.flagged)}/{int(row.capacity)}" for row in periods.itertuples())}.

## لماذا اخترت هذه العتبة؟
{answer('threshold_reason')}

## الخسارة والسعة
{answer('cost_capacity_tradeoff')}

## فرق المناطق وما يحتاج إلى مراجعة
{answer('regional_gap')}

## حدود النتيجة
{answer('limitations')}

OOF تغطي {coverage['whole_data_coverage']:.2%} من التدريب و100% من الصفوف المؤهلة؛ {coverage['warmup_without_oof']:,} صفًا تمهيديًا بلا تنبؤ. اختيار العتبة وتقدير خسارتها هنا يستخدمان أهدافOOF نفسها؛ هذه نتيجة تطوير لا اختبار نهائي. لم نستخدم التحدي. المقارنة الجغرافية وصفية وليست شهادة عدالة، والأوزان لا تضمن معايرة الدرجات.

## سؤالك الأول: لماذا قد تخدعكAccuracy؟
{answer('why_accuracy_misleads')}

## سؤالك الثاني: لماذا تختار علىOOF؟
{answer('why_oof')}

أدلتك في `artifacts/threshold_metrics.json` و`day3_period_capacity.csv` و`day3_region_audit.csv` و`day3_cost_sensitivity.csv` و`cost_curve.png`. الحساسية سيناريوهات ±20% لخسارةFN، وليست فترات ثقة. راجع السعة والمعايرة عند تغير البيانات؛ لا تفترض ثباتهما مستقبلًا.
"""
(REPORTS / "DECISION_CARD.md").write_text(card, encoding="utf-8")
artifact_files = list(csv_outputs) + list(json_outputs) + ["cost_curve.png", "day3_roc_pr.png", "day3_capacity_regions.png", "environment.json"]
with zipfile.ZipFile(ARTIFACTS / "day3_artifacts.zip", "w", zipfile.ZIP_DEFLATED) as bundle:
    for filename in artifact_files:
        bundle.write(ARTIFACTS / filename, "artifacts/" + filename)
    bundle.write(REPORTS / "DECISION_CARD.md", "reports/DECISION_CARD.md")
print(f"{status} | {checkpoint['status']} | {SOURCE} | {run['elapsed_seconds']} seconds | CPU")
print(f"Saved {len(artifact_files)+1} files in day3_artifacts.zip — complete your reasoning and download.")

## قبل أن تنتقل
أشر إلىFN وFP والمقامات، ثم اشرح لماذا تختلف أفضل عتبة دون قيد عن العتبة التي تلائم الفريق. اعرض أعداد كل فترة؛ لا تكتفِ بنسبة مجمعة. قارن الوزن وإعادة العينات دون فرض فائز. اكتب حدود المعايرة والتغطية واختيار العتبة على بيانات التطوير.

إذا تعثرت الجلسة، احفظ ما لديك وأعد علىCPU المجاني لاحقًا. المثال المحفوظ يتيح النقاش ولا يستبدل تشغيلك. [دليل اليوم الثالث](https://github.com/almiyead-rgb/sda-dsc-211-student-template/blob/main/DAY3_GUIDE.md).

**المراجع:** [أوزانLightGBM](https://lightgbm.readthedocs.io/en/v4.6.0/Parameters.html#scale_pos_weight) · [ضبط عتبة القرار](https://scikit-learn.org/stable/modules/classification_threshold.html) · [Average Precision](https://scikit-learn.org/1.6/modules/generated/sklearn.metrics.average_precision_score.html).

Next, inspect explanations and calibration. Today's scores and threshold are development evidence, not a final decision for real people.